# BERTs adicionales (5 folds, OOF) — ap-2026-q-2
Funciona en **Kaggle Notebooks** (recomendado: 30 h/semana de GPU gratis) o en **Colab**.

**Kaggle:** New Notebook → File → Import Notebook (este archivo) → *Add Input*: competencia `ap-2026-q-2` → Settings: **Accelerator = GPU T4 x2**, **Internet = On** → **Save Version → Save & Run All (Commit)**. Corre en segundo plano aunque cierres la pestaña; al terminar, los `.npy` quedan en la pestaña *Output* de la versión.

**Colab:** Entorno → GPU T4 → Ejecutar todo. Lee los datos de la raíz de *Mi unidad* y guarda en *Mi unidad/bert_out/* fold por fold (reanudable).

Entrena 3 modelos distintos (diversidad para el ensamble): BERTIN (RoBERTa en español), BETO uncased y BETO cased con otra entrada/semilla.

In [ ]:
import os, glob
KAGGLE = os.path.exists("/kaggle/input")
if KAGGLE:
    ROOT = os.path.dirname(glob.glob("/kaggle/input/**/train_data.jsonlines", recursive=True)[0]); OUT = "/kaggle/working"
!pip -q install sentencepiece protobuf
if not KAGGLE:
    from google.colab import drive
    drive.mount('/content/drive'); ROOT = "/content/drive/MyDrive"; OUT = f"{ROOT}/bert_out"
print(ROOT, OUT)

In [ ]:
import json, os, re, time, numpy as np, torch
from sklearn.model_selection import StratifiedKFold
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
os.makedirs(OUT, exist_ok=True)
load = lambda p: [json.loads(l) for l in open(p, encoding="utf-8")]
tr, te = load(f"{ROOT}/train_data.jsonlines"), load(f"{ROOT}/test_data.jsonlines")
y = np.array([r["condition"] == "used" for r in tr]).astype(int); dev = "cuda"
assert torch.cuda.is_available(), "Activar GPU (Kaggle: Settings -> Accelerator; Colab: Entorno -> T4 GPU)"

def txt_rich(r):  # igual a colab_bert_v2
    at = " ; ".join(f"{a.get('name','')}: {a.get('value_name') or ''}" for a in (r.get("attributes") or [])[:8])
    pay = ",".join(p.get("description", "") for p in (r.get("non_mercado_pago_payment_methods") or []))
    sh = r.get("shipping") or {}
    return (f"{r['title']} | {r['category_id']} | {r['listing_type_id']} | precio {r.get('price')} | cantidad {r.get('initial_quantity')} "
            f"| garantia: {(r.get('warranty') or 'ninguna')[:50]} | envio {sh.get('mode')} gratis {sh.get('free_shipping')} | pagos: {pay} | {at}")

def txt_alt(r):  # entrada distinta: garantia completa primero, variaciones, vendidos, tags
    sh = r.get("shipping") or {}
    var = " ; ".join(f"{c.get('name','')}: {c.get('value_name') or ''}" for v in (r.get("variations") or [])[:3] for c in (v.get("attribute_combinations") or []))
    return (f"garantia: {(r.get('warranty') or 'ninguna')[:120]} | {r['title']} | vendidos {r.get('sold_quantity')} de {r.get('initial_quantity')} "
            f"| disponibles {r.get('available_quantity')} | {r['listing_type_id']} | {r.get('buying_mode')} | fotos {len(r.get('pictures') or [])} "
            f"| retiro {sh.get('local_pick_up')} | tags {' '.join(r.get('tags') or [])} | {var}")

# (nombre, modelo, lr, funcion de texto, semilla)
MODELS = [("bertin", "bertin-project/bertin-roberta-base-spanish", 3e-5, txt_rich, 0),
          ("beto_unc", "dccuchile/bert-base-spanish-wwm-uncased", 3e-5, txt_rich, 1),
          ("beto_alt", "dccuchile/bert-base-spanish-wwm-cased", 3e-5, txt_alt, 2)]
EPOCHS, BS, MAXLEN = 3, 64, 128

In [ ]:
def run(name, MODEL, LR, fx, seed):
    if os.path.exists(f"{OUT}/test_{name}.npy"): print(name, "ya estaba"); return
    Xtr, Xte = [fx(r) for r in tr], [fx(r) for r in te]
    tok = AutoTokenizer.from_pretrained(MODEL)
    Itr = tok(Xtr, truncation=True, max_length=MAXLEN)["input_ids"]; Ite = tok(Xte, truncation=True, max_length=MAXLEN)["input_ids"]; pad = tok.pad_token_id
    def batch(I, idx):
        L = max(len(I[i]) for i in idx); ids = torch.full((len(idx), L), pad)
        for k, i in enumerate(idx): ids[k, :len(I[i])] = torch.tensor(I[i])
        return ids.to(dev), (ids != pad).long().to(dev)
    def predict(model, I, idx):
        model.eval(); out = []; order = sorted(idx, key=lambda i: len(I[i]))
        with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16):
            for s in range(0, len(order), 256):
                ids, m = batch(I, order[s:s + 256]); out.append(torch.softmax(model(input_ids=ids, attention_mask=m).logits.float(), -1)[:, 1].cpu().numpy())
        p = np.concatenate(out); res = np.zeros(len(idx)); pos = {i: k for k, i in enumerate(idx)}
        for o, v in zip(order, p): res[pos[o]] = v
        return res
    t0 = time.time(); yt = torch.tensor(y)
    for k, (a, b) in enumerate(StratifiedKFold(5, shuffle=True, random_state=42).split(Xtr, y)):
        fp = f"{OUT}/{name}_fold{k}.npz"
        if os.path.exists(fp): continue
        torch.manual_seed(100 * seed + k); model = AutoModelForSequenceClassification.from_pretrained(MODEL, num_labels=2).to(dev)
        opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01); steps = EPOCHS * ((len(a) + BS - 1) // BS)
        sch = get_linear_schedule_with_warmup(opt, int(.06 * steps), steps); scaler = torch.amp.GradScaler()
        for ep in range(EPOCHS):
            model.train(); perm = np.random.RandomState(1000 * seed + 10 * k + ep).permutation(a)
            for s in range(0, len(perm), BS):
                i = list(perm[s:s + BS]); ids, m = batch(Itr, i)
                with torch.autocast("cuda", dtype=torch.float16): loss = model(input_ids=ids, attention_mask=m, labels=yt[i].to(dev)).loss
                opt.zero_grad(); scaler.scale(loss).backward(); scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(opt); scaler.update(); sch.step()
        ob = predict(model, Itr, list(b)); tb = predict(model, Ite, list(range(len(Xte))))
        np.savez(fp, idx=b, oof=ob, test=tb)
        print(f"{name} fold {k} acc {((ob > .5) == y[b]).mean():.4f} ({time.time() - t0:.0f}s)", flush=True)
        del model, opt; torch.cuda.empty_cache()
    oof = np.zeros(len(y)); tp = np.zeros(len(te))
    for k in range(5):
        z = np.load(f"{OUT}/{name}_fold{k}.npz"); oof[z["idx"]] = z["oof"]; tp += z["test"] / 5
    print(name, "OOF acc", ((oof > .5) == y).mean()); np.save(f"{OUT}/oof_{name}.npy", oof); np.save(f"{OUT}/test_{name}.npy", tp)

for args in MODELS: run(*args)
print("TODO LISTO ->", sorted(os.listdir(OUT)))